# Reading the data in a CSV file via DictReader as a dict()

In [1]:
import nltk, string
#nltk.download('punkt')
#from pattern3.text.en import lemma
from nltk import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from nltk.stem import WordNetLemmatizer
import spacy, gensim

ModuleNotFoundError: No module named 'spacy'

In [2]:
!pip install spacy

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\programdata\\anaconda3\\lib\\site-packages\\__pycache__\\typing_extensions.cpython-38.pyc'
Consider using the `--user` option or check the permissions.




  Attempting uninstall: typing-extensions
    Found existing installation: typing-extensions 3.7.4.3
    Uninstalling typing-extensions-3.7.4.3:


In [5]:
from csv import DictReader
def read_csv(file_name):

    rows = []
    with open(file_name,'r',encoding = 'utf-8') as table:
        
        r = DictReader(table)
        for line in r:
            rows.append(line)
    return rows

In [6]:
import pandas as pd
file_name = 'nytimes_articles.csv' #the csv file name or directory
dataset = read_csv(file_name)
 
type(dataset) 
#dataset[:5]

list

In [66]:
data_set = [data['0'] for data in dataset] # the input to the vectorizer should be a list of documents
#data_set[:5]

## Creating a list of stop words, combined from a number of different sources 

In [7]:
# function to check if a line is blank
def isLineEmpty(line):
    return len(line.strip()) == 0

In [9]:
custom_stop_words = []
with open( "newStopwords.txt", "r" ) as fin:
    for line in fin.readlines():
        if not isLineEmpty(line):
            custom_stop_words.append( line.strip() )
# note that we need to make it hashable
print("Stopword list has %d entries" % len(custom_stop_words) )
#custom_stop_words

Stopword list has 599 entries


# Topic Modeling using CountVectorizer and LDA

In [10]:
import lda 
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text 
#stop_words = text.ENGLISH_STOP_WORDS.union(my_additional_stop_words)

n_topics = 14 # the maximum number of topics to be considerd by LDA
n_iter = 500 # number of iterations


ModuleNotFoundError: No module named 'lda'

## Tokenize the text

In [ ]:
def tokenized(texts):
    for txt in texts:
        yield(gensim.utils.simple_preprocess(str(txt), deacc=True))  # deacc=True removes punctuations

data_words = list(tokenized(data_set))

#print(data_words[:1])

## Lemmatize tokenized words, keeping only noun, adjective, verb, and adverb words

In [ ]:
def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent)) 
        texts_out.append(" ".join([token.lemma_ if token.lemma_ not in ['-PRON-'] else '' for token in doc if token.pos_ in allowed_postags]))
    return texts_out

# Initialize spacy 'en' model, keeping only tagger component (for efficiency)
# Run in terminal: python3 -m spacy download en
nlp = spacy.load('en', disable=['parser', 'ner'])

# Do lemmatization keeping only Noun, Adj, Verb, Adverb
data_lemmatized = lemmatization(data_words, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV'])

#print(data_lemmatized[:1])

## Vectorize lemmatizied text using CountVectorizer

In [74]:
vectorizer = CountVectorizer(analyzer='word',       
                             min_df=5,                        # minimum reqd occurences of a word 
                             stop_words=custom_stop_words,   # remove stop words
                             lowercase=True,                 # convert all words to lowercase
                             token_pattern='[a-zA-Z0-9]{3,}'  # num chars > 3
                            )

data_vectorized = vectorizer.fit_transform(data_lemmatized)

In [ ]:
#using latentDirichletAllocation from sklearn.decomposition  
#lda_model2 = LatentDirichletAllocation(n_components=14,               # Number of topics
#                                      max_iter=500,               # Max learning iterations
#                                      learning_method='online',   
#                                      random_state=123,          # Random state
#                                      batch_size=128,            # n docs in each learning iter
#                                      evaluate_every = -1,       # compute perplexity every n iters, default: Don't
#                                      n_jobs = -1               # Use all available CPUs
#                                     )
#lda2_output = lda_model2.fit_transform(data_vectorized)
#print(lda_model2)  # Model attributes

## Run the LDA function using 14 topics and 5000 iterations

In [77]:
lda_model = lda.LDA(n_topics=n_topics,n_iter=n_iter)
x_topics = lda_model.fit_transform(data_vectorized )

INFO:lda:n_documents: 8885
INFO:lda:vocab_size: 24177
INFO:lda:n_words: 3011605
INFO:lda:n_topics: 14
INFO:lda:n_iter: 500
/home/ubuntu/.local/lib/python3.7/site-packages/lda/utils.py:55: FutureWarning: Conversion of the second argument of issubdtype from `int` to `np.signedinteger` is deprecated. In future, it will be treated as `np.int64 == np.dtype(int).type`.
  if sparse and not np.issubdtype(doc_word.dtype, int):
INFO:lda:<0> log likelihood: -34987174
INFO:lda:<10> log likelihood: -28792857
INFO:lda:<20> log likelihood: -27261898
INFO:lda:<30> log likelihood: -26889708
INFO:lda:<40> log likelihood: -26727878
INFO:lda:<50> log likelihood: -26625813
INFO:lda:<60> log likelihood: -26560379
INFO:lda:<70> log likelihood: -26516051
INFO:lda:<80> log likelihood: -26483904
INFO:lda:<90> log likelihood: -26460445
INFO:lda:<100> log likelihood: -26440291
INFO:lda:<110> log likelihood: -26425486
INFO:lda:<120> log likelihood: -26415368
INFO:lda:<130> log likelihood: -26409674
INFO:lda:<140> 

## Reduction of the dimensionality from 14 -> 2 using t-SNE

In [78]:
from sklearn.manifold import TSNE

tsne_model = TSNE (n_components = 2, verbose = 1, random_state = 0, angle = 0.99, init='pca')
tsne_lda = tsne_model.fit_transform(lda_output)

[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 8885 samples in 0.017s...
[t-SNE] Computed neighbors for 8885 samples in 1.806s...
[t-SNE] Computed conditional probabilities for sample 1000 / 8885
[t-SNE] Computed conditional probabilities for sample 2000 / 8885
[t-SNE] Computed conditional probabilities for sample 3000 / 8885
[t-SNE] Computed conditional probabilities for sample 4000 / 8885
[t-SNE] Computed conditional probabilities for sample 5000 / 8885
[t-SNE] Computed conditional probabilities for sample 6000 / 8885
[t-SNE] Computed conditional probabilities for sample 7000 / 8885
[t-SNE] Computed conditional probabilities for sample 8000 / 8885
[t-SNE] Computed conditional probabilities for sample 8885 / 8885
[t-SNE] Mean sigma: 0.059605
[t-SNE] KL divergence after 250 iterations with early exaggeration: 78.904213
[t-SNE] KL divergence after 1000 iterations: 1.363574


## Set up for a Bokeh plot

In [80]:
_lda_keys = []
for i in range(x_topics.shape[0]):
 _lda_keys +=  x_topics[i].argmax(),

In [81]:
topic_summaries = []
topic_word = lda_model.topic_word_  # all topic words
vocab = cvectorizer.get_feature_names()
for i, topic_dist in enumerate(topic_word):
 topic_words = np.array(vocab)[np.argsort(topic_dist)][:-(n_top_words + 1):-1] # get!
 topic_summaries.append(' '.join(topic_words)) # append!

In [82]:
title = 'news sample topics'
num_example = len(x_topics)

plot_lda = bp.figure(plot_width=1400, plot_height=1100,
                    title=title,
                    tools='pan,wheel_zoom,box_zoom,reset,hover,previewsave',
                    x_axis_type=None, y_axis_type=None, min_border=1)


ys=tsne_lda[:, 1]
xs=tsne_lda[:, 0]
source = bp.ColumnDataSource(data = dict(
                x = xs,
                y = ys,
                color=colormap[_lda_keys][:num_example],
                content = data_set[:num_example],
                topic_key = _lda_keys[:num_example]
                  ))
plot_lda.scatter(x ='x',
                 y ='y', 
                source=source,
                color ='color')

GlyphRenderer(id='1875', ...)

## Plot the LDA data and save it in an HTML file

In [83]:
# randomly choose a news (within a topic) coordinate as the crucial words coordinate
topic_coord = np.empty((x_topics.shape[1], 2)) * np.nan
for topic_num in _lda_keys:
 if not np.isnan(topic_coord).any():
   break
 topic_coord[topic_num] = tsne_lda[_lda_keys.index(topic_num)]

# plot crucial words
for i in range(x_topics.shape[1]):
 plot_lda.text(topic_coord[i, 0], topic_coord[i, 1], [topic_summaries[i]])

# hover tools
hover = plot_lda.select(dict(type=HoverTool))
hover.tooltips = {"content": "@content - topic: @topic_key"}

# save the plot
save(plot_lda, '{}.html'.format(title))

/home/ubuntu/.local/lib/python3.7/site-packages/bokeh/io/saving.py:126: UserWarning: save() called but no resources were supplied and output_file(...) was never called, defaulting to resources.CDN
  warn("save() called but no resources were supplied and output_file(...) was never called, defaulting to resources.CDN")
/home/ubuntu/.local/lib/python3.7/site-packages/bokeh/io/saving.py:139: UserWarning: save() called but no title was supplied and output_file(...) was never called, using default title 'Bokeh Plot'
  warn("save() called but no title was supplied and output_file(...) was never called, using default title 'Bokeh Plot'")


'/home/ubuntu/project/news sample topics.html'